# Retail Customer Segmentation & Sales Analysis
End‑to‑end analysis using a 100K+ transaction dataset.

## Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [ ]:
df = pd.read_excel('../data/Online Retail.xlsx')
df.head()

## Data Overview

In [ ]:
df.info()
df.describe()

## Data Cleaning

In [ ]:
# remove invalid rows
df = df[df['Quantity'] > 0]
df = df.dropna(subset=['CustomerID'])

# revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

df.head()

## KPI Metrics

In [ ]:
revenue = df['Revenue'].sum()
orders = df['InvoiceNo'].nunique()
customers = df['CustomerID'].nunique()
aov = df.groupby('InvoiceNo')['Revenue'].sum().mean()

revenue, orders, customers, aov

## Revenue by Country

In [ ]:
country_rev = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_rev.plot(kind='bar')
plt.title('Revenue by Country')
plt.show()

## Monthly Revenue Trend

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.to_period('M')

monthly = df.groupby('Month')['Revenue'].sum()
monthly.plot()
plt.title('Monthly Revenue Trend')
plt.show()

## RFM Segmentation

In [ ]:
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']

rfm['R'] = pd.qcut(rfm['Recency'],4,labels=[4,3,2,1])
rfm['F'] = pd.qcut(rfm['Frequency'],4,labels=[1,2,3,4])
rfm['M'] = pd.qcut(rfm['Monetary'],4,labels=[1,2,3,4])

rfm['Score'] = rfm[['R','F','M']].astype(int).sum(axis=1)

def segment(s):
    if s>=10: return 'High Value'
    elif s>=7: return 'Loyal'
    elif s>=5: return 'Potential'
    else: return 'At Risk'

rfm['Segment'] = rfm['Score'].apply(segment)
rfm.head()

## Segment Distribution

In [ ]:
rfm['Segment'].value_counts().plot(kind='bar'); plt.title('Customer Segments'); plt.show()